# Module 5: Server Features — Long-Running Jobs, Crons, Durable Execution, Streaming & HITL

> Part of the **Modular Workshops** series. Standalone, ~25 min.

In the deploy module you shipped the in-store shopping assistant to a LangGraph server (`langgraph dev` locally, LangSmith Deployments in the cloud). That server isn't just an HTTP wrapper around `agent.invoke` — it's a **runtime** with production features you get for free, no extra code in the agent:

1. **Long-running / background jobs** — kick off a run and return immediately; the server keeps working.
2. **Crons** — schedule the assistant to run on a recurring basis (e.g. a nightly restock check).
3. **Durable execution** — runs are checkpointed to a thread, so they survive disconnects and can be resumed.
4. **Streaming** — stream tokens, state updates, and tool calls as the run happens.
5. **Human-in-the-loop (HITL)** — the run pauses on an interrupt, waits for a human decision, then resumes exactly where it left off.
6. **A/B testing with assistants** — run two prompt variations as two assistants on the same graph, and route live traffic between them.
7. **Events out** — webhooks, automation rules, and alerts that notify *your* systems when a run finishes, a trace pattern appears, a threshold breaches, or a prompt changes.

Everything below talks to the **same local server** through the LangGraph SDK (`langgraph_sdk`). We drive the deployed `deep_agent` — our shopping assistant — so you see the features against a real graph, not a toy.

> These are LangGraph **Server** features. They work identically against a cloud **LangSmith Deployment**: point the SDK at the deployment URL instead of `localhost`.

## Setup — launch the local server

This module starts its own `langgraph dev` server in the background (in-memory, no Docker) and connects to it. `langgraph dev` reads the workshop's `langgraph.json`, which registers the shopping assistant as the `deep_agent` graph.

The cell below:
- starts `langgraph dev --no-browser --no-reload` on port **2024** (reusing an already-running server if one is up),
- polls the health endpoint until it's ready,
- opens an SDK client against it.

> If you'd rather run the server yourself, open a terminal at the workshop root and run `langgraph dev`, then skip straight to the `get_sync_client(...)` line.

In [ ]:
import sys, os, time, socket, subprocess, atexit, urllib.request
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

PORT = 2024
BASE_URL = f"http://localhost:{PORT}"

def _server_up(url, timeout=1.0):
    """Return True if the local LangGraph server answers its health check."""
    try:
        with urllib.request.urlopen(f"{url}/ok", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

dev_proc = None
if _server_up(BASE_URL):
    print(f"Reusing a langgraph dev server already running on {BASE_URL}")
else:
    print(f"Starting `langgraph dev` on {BASE_URL} ...")
    # --no-reload keeps a single stable process; logs go to server.log at the root.
    log = open(project_root / "langgraph_dev.log", "w")
    dev_proc = subprocess.Popen(
        ["langgraph", "dev", "--no-browser", "--no-reload", "--port", str(PORT)],
        cwd=str(project_root), stdout=log, stderr=subprocess.STDOUT,
    )
    # Make sure we don't leave an orphaned server behind when the kernel exits.
    atexit.register(lambda: dev_proc and dev_proc.poll() is None and dev_proc.terminate())

    for _ in range(60):  # wait up to ~60s for startup (first boot imports the graph)
        if _server_up(BASE_URL):
            break
        if dev_proc.poll() is not None:
            raise RuntimeError("langgraph dev exited early — see langgraph_dev.log")
        time.sleep(1)
    else:
        raise TimeoutError("langgraph dev did not become healthy in time — see langgraph_dev.log")
    print("Server is up.")

In [ ]:
from langgraph_sdk import get_sync_client
from langgraph_sdk.schema import Command

client = get_sync_client(url=BASE_URL)

# The graph id is whatever `langgraph.json` registered — here, our shopping assistant.
GRAPH_ID = "deep_agent"

# Find the assistant for our graph. If a *different* langgraph dev server was
# already running on this port, it won't have `deep_agent` — fail loudly rather
# than silently driving the wrong graph.
matches = client.assistants.search(graph_id=GRAPH_ID, limit=1)
if not matches:
    raise RuntimeError(
        f"Connected to {BASE_URL}, but it does not serve the '{GRAPH_ID}' graph. "
        "Another langgraph dev server may be running on this port. Stop it (or set a "
        "different PORT above) and re-run — this notebook needs the workshop's own server."
    )
assistant_id = matches[0]["assistant_id"]
print("Connected. Assistant:", assistant_id, "(graph:", GRAPH_ID + ")")

## 1. Long-running / background jobs

Not every request should block. A shopper might ask the assistant to plan a whole week of dinners and build the aisle-by-aisle route — that's many tool calls and can take a while. The server lets you **start a run in the background** and get a run id back immediately, then check on it later.

The pattern is a **thread** (a durable conversation) + a **run** (one execution on that thread). Creating a run with `client.runs.create(...)` returns right away with a `pending`/`running` status; the server keeps executing.

In [ ]:
# Create a durable thread to run on, then kick off a background run.
thread = client.threads.create()
print("thread:", thread["thread_id"])

bg_run = client.runs.create(
    thread["thread_id"],
    assistant_id,
    input={"messages": [{"role": "user", "content": (
        "Plan 3 quick weeknight dinners, then build an aisle-ordered shopping "
        "route for everything I need. Take your time."
    )}]},
)
print("run:", bg_run["run_id"], "->", bg_run["status"])
print("Returned immediately — the server keeps working in the background.")

### Poll or join the background run

You can poll the run's status, or **join** it — a blocking call that waits for the run to finish and returns its final output. `join` is handy when you started work in the background and now want the result.

In [ ]:
# Option A: poll the status without blocking.
status = client.runs.get(thread["thread_id"], bg_run["run_id"])["status"]
print("polled status:", status)

# Option B: join — block until the run completes and return its final state.
final = client.runs.join(thread["thread_id"], bg_run["run_id"])
last = final["messages"][-1]
print("\nFinal answer:\n", (last.get("content") or "")[:800])

## 2. Crons — scheduled runs

The server can trigger the assistant on a **schedule** with no external scheduler. For a store, that might be a nightly job that re-checks stock on the items a shopper flagged, or a Monday-morning "here's your weekly staples list" run.

`client.crons.create(...)` takes a standard cron expression and the input to run each time. A cron with no thread starts a fresh stateless run on each fire; `crons.create_for_thread(...)` runs on a specific thread so state accumulates.

> Crons run **on the server**, so they only fire while the server (or cloud deployment) is up. On a real LangSmith Deployment this runs 24/7; against `langgraph dev` it fires while your dev server is running.

In [ ]:
# Schedule a nightly restock check at 6:00 AM every day.
# Cron format: minute hour day-of-month month day-of-week
cron = client.crons.create(
    assistant_id,
    schedule="0 6 * * *",
    input={"messages": [{"role": "user", "content": (
        "Nightly restock check: is ground beef back in stock yet? "
        "If so, tell me its aisle and price; if not, say so."
    )}]},
)
print("Created cron:", cron["cron_id"], "schedule:", cron["schedule"])

# List the crons registered on this server.
for c in client.crons.search(assistant_id=assistant_id, limit=10):
    print(f"  {c['cron_id']}  {c['schedule']}")

We won't wait for 6 AM — delete the cron so it doesn't fire on every workshop run. In production you'd leave it in place (or manage it in the LangSmith UI).

In [ ]:
client.crons.delete(cron["cron_id"])
print("Deleted cron", cron["cron_id"])

## 3. Durable execution

Every run executes **on a thread**, and the server **checkpoints** thread state after each step. That's what makes runs durable: if a client disconnects, the run keeps going server-side, and the thread remembers everything — message history, files the agent wrote, tool results.

Two things fall out of this for free:
- **Resilience** — a dropped connection doesn't lose work; re-attach with `runs.join`.
- **Memory across runs** — a follow-up run on the *same thread* sees the earlier turns. Below, we ask a follow-up that only makes sense if the server retained the previous context.

In [ ]:
# Reuse the SAME thread from section 1. The assistant already planned dinners
# and a route there — a follow-up should build on that without us re-sending it.
followup = client.runs.wait(
    thread["thread_id"],           # same durable thread
    assistant_id,
    input={"messages": [{"role": "user", "content": (
        "From that plan, which single aisle has the most of my items?"
    )}]},
)
print("Follow-up answer (server retained the earlier plan):\n")
print((followup["messages"][-1].get("content") or "")[:600])

# Inspect the durable state the server kept for this thread.
state = client.threads.get_state(thread["thread_id"])
msg_count = len(state["values"].get("messages", []))
print(f"\nThread now holds {msg_count} messages across multiple runs — all checkpointed server-side.")

## 4. Streaming

Instead of waiting for the whole run, **stream** it. The server emits events as they happen — useful for showing a shopper progress ("checking the directory… found it in aisle 6") instead of a spinner.

`stream_mode` controls what you get:
- `"messages"` — token-by-token LLM output (great for chat UIs).
- `"updates"` — the state delta after each node/step (great for showing tool calls and progress).
- `"values"` — the full state after each step.
- `"events"` — low-level LangChain events.

Below we stream `"updates"` on a fresh thread so you can watch the assistant work step by step.

In [ ]:
def _text_of(content):
    """Pull just the human-readable text out of a message's content, which the
    Responses API returns as a list of blocks (reasoning/tool/text), not a str."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content
                       if isinstance(b, dict) and b.get("type") == "text")
    return ""

stream_thread = client.threads.create()

print("Streaming updates as the assistant works:\n")
for chunk in client.runs.stream(
    stream_thread["thread_id"],
    assistant_id,
    input={"messages": [{"role": "user", "content": (
        "What aisle is the salsa in, and how much is it?"
    )}]},
    stream_mode="updates",
):
    if chunk.event != "updates" or not chunk.data:
        continue
    for node, update in chunk.data.items():
        if not isinstance(update, dict):
            continue
        for m in update.get("messages", []):
            # Tool calls the model decided to make.
            for tc in (m.get("tool_calls") or []):
                print(f"  [{node}] tool call -> {tc['name']}({tc.get('args')})")
            # Tool results and final text (skip the reasoning/encrypted blocks).
            text = _text_of(m.get("content"))
            if text.strip():
                print(f"  [{node}] {text[:200]}")

You can also stream **tokens** as the model generates them — set `stream_mode="messages"` and print each chunk's text. This is what powers a live typing effect in a chat UI.

In [ ]:
def _text_of(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content
                       if isinstance(b, dict) and b.get("type") == "text")
    return ""

print("Token stream:\n")
shown = ""   # messages/partial sends CUMULATIVE text, so print only what's new
for chunk in client.runs.stream(
    stream_thread["thread_id"],
    assistant_id,
    input={"messages": [{"role": "user", "content": "In one sentence, what makes a good taco?"}]},
    stream_mode="messages",
):
    # The event is "messages/partial" (token updates) / "messages/metadata".
    if chunk.event == "messages/partial" and isinstance(chunk.data, list) and chunk.data:
        full = _text_of(chunk.data[0].get("content"))
        if full and full.startswith(shown):
            print(full[len(shown):], end="", flush=True)  # new tokens only
            shown = full
        elif full:
            print(full, end="", flush=True)
            shown = full
print()

## 5. Human-in-the-loop (HITL)

The shopping assistant is deployed with an **interrupt** on file writes (`interrupt_on={"write_file": True, "edit_file": True}` in `agents/deep_agent/agent.py`). So when the assistant tries to save your shopping list to a file, the run **pauses** and waits for a human to approve — the server holds the checkpoint until you respond.

This is durable execution + interrupts working together: the run can sit paused indefinitely, and resuming continues from the exact step, on the server.

**Step 1 — start a run that will try to write a file.** The run comes back with status `interrupted` and the pending action to approve.

In [ ]:
hitl_thread = client.threads.create()

interrupted = client.runs.wait(
    hitl_thread["thread_id"],
    assistant_id,
    input={"messages": [{"role": "user", "content": (
        "Save my taco shopping list (tortillas, salsa, shredded cheese) to /shopping_list.md"
    )}]},
)

# When a run interrupts, the SDK surfaces the pending request under __interrupt__.
interrupts = interrupted.get("__interrupt__") or []
if interrupts:
    print("Run PAUSED for human approval. Pending action(s):")
    for it in interrupts:
        val = it["value"] if isinstance(it, dict) else it
        for req in (val.get("action_requests", []) if isinstance(val, dict) else []):
            print(f"  - {req['name']}  args={req.get('args')}")
    print("\nThe server is holding this run's checkpoint until we decide.")
else:
    print("No interrupt surfaced — check that the assistant has interrupts enabled.")

**Step 2 — resume with a human decision.** We approve the write by resuming the same run with `Command(resume=...)`. The server picks the run back up from the checkpoint and finishes it. (You could also `reject` or `edit` the action here.)

In [ ]:
resumed = client.runs.wait(
    hitl_thread["thread_id"],
    assistant_id,
    command=Command(resume={"decisions": [{"type": "approve"}]}),
)

print("Resumed and completed. Final answer:\n")
print((resumed["messages"][-1].get("content") or "")[:500])

# Confirm the file the assistant wrote now lives in the thread's durable state.
files = client.threads.get_state(hitl_thread["thread_id"])["values"].get("files", {})
print("\nFiles written to the thread:", list(files))

## 6. A/B testing with assistants

The team wants to A/B test prompt variations live. The supported way to do this on the platform is **assistants**: *one deployed graph, multiple assistants, each holding its own config* (prompt, model, tools), each independently versioned with promote / roll back. The docs call out A/B testing and gradual rollout as the intended use cases.

What this changes versus a hand-rolled variant flag:

- **Assistants replace the flag, not the measurement.** Variant A and variant B become two `assistant_id`s on the same `graph_id`. No code change to switch a prompt, no redeploy — the config lives *outside* the graph. That's the "a non-engineer owns the prompt" story: they edit an assistant's config in the UI/SDK, not the codebase.
- **Something still has to route each request to an assistant.** There is no traffic splitter in the product — that's a line in *your* router or app. We show a tiny one below.
- **Measurement is the same move:** group the dashboards by the thing that distinguishes the runs. Group by `assistant_id`, and — belt and braces — also stamp a `variant` tag in each run's `metadata` so you can group by that too if `assistant_id` isn't a convenient grouping path in your project.

Our `deep_agent` factory reads an optional `system_prompt` from `config.configurable` (see `agents/deep_agent/agent.py`), so each assistant can pin its own prompt variation with **no graph change**.

### 6.1 Create two assistants — the two variants

Two prompt variations of the same shopping assistant:

- **Variant A (control)** — the current, neutral prompt.
- **Variant B (candidate)** — a "concierge" prompt that always states stock up front and proactively offers a substitution when something's out.

Each is a separate `assistant_id` on the same `deep_agent` graph, tagged with `variant` metadata. We use a stable `assistant_id` + `if_exists="do_nothing"` so re-running the notebook is idempotent.

In [ ]:
import uuid as _uuid

# Deterministic ids so re-runs update the same two assistants instead of piling up.
def _stable_id(name):
    return str(_uuid.uuid5(_uuid.NAMESPACE_DNS, f"workshop-ab-{name}"))

VARIANT_PROMPTS = {
    "A": "You are an expert in-store shopping assistant.",
    "B": ("You are a concierge in-store shopping assistant. For every item, state its "
          "stock status FIRST, then the aisle and price. If an item is out of stock, "
          "proactively suggest one in-store substitution without being asked."),
}

variants = {}
for label, prompt in VARIANT_PROMPTS.items():
    a = client.assistants.create(
        GRAPH_ID,
        assistant_id=_stable_id(label),
        config={"configurable": {"system_prompt": prompt}},
        metadata={"variant": label},
        name=f"shopping-variant-{label}",
        if_exists="do_nothing",   # idempotent across re-runs
    )
    variants[label] = a["assistant_id"]
    print(f"variant {label}: {a['assistant_id']}  (name={a['name']})")

print("\nTwo assistants, one graph — each holds its own prompt, no redeploy.")

### 6.2 Version a variant — promote / roll back

Assistants are independently **versioned**. Editing a variant's config with `assistants.update(...)` creates a **new version**; `assistants.set_latest(...)` promotes a specific version (or rolls back to an earlier one). This is how a prompt owner ships and reverts a change without touching code or redeploying.

In [ ]:
ab_id = variants["B"]

# Edit variant B's prompt -> creates version 2.
client.assistants.update(
    ab_id,
    config={"configurable": {"system_prompt": VARIANT_PROMPTS["B"] + " Keep replies under 4 lines."}},
)
versions = client.assistants.get_versions(ab_id)
print("variant B versions:", [v["version"] for v in versions])

# Promote (or roll back) to a specific version. Here we roll B back to v1.
client.assistants.set_latest(ab_id, 1)
current = client.assistants.get(ab_id)
print("variant B active version is now:", current["version"])

### 6.3 Route requests to A or B

The platform won't split traffic for you — routing is a line in your app. A common pattern is a **stable hash of the shopper id** so a given shopper always sees the same variant (sticky assignment), with a split ratio you control. We also stamp `variant` metadata on the run so the run is groupable by variant even where `assistant_id` isn't.

In [ ]:
import hashlib

def pick_variant(shopper_id: str, split_b: float = 0.5) -> str:
    """Stable A/B assignment: hash the shopper id to [0,1) and compare to the split."""
    h = int(hashlib.sha256(shopper_id.encode()).hexdigest(), 16) % 10_000
    return "B" if (h / 10_000) < split_b else "A"

def ask_ab(shopper_id: str, message: str, split_b: float = 0.5):
    """Route one request to the assigned variant and return (variant, answer)."""
    label = pick_variant(shopper_id, split_b)
    result = client.runs.wait(
        None,                       # stateless run; a thread would make it durable
        variants[label],            # <-- the assistant_id is the variant
        input={"messages": [{"role": "user", "content": message}]},
        # Belt-and-braces: also stamp the variant on the run's metadata so
        # dashboards can group by metadata.variant OR by assistant_id.
        metadata={"variant": label, "shopper_id": shopper_id, "experiment": "shopping-prompt-ab"},
    )
    answer = result["messages"][-1].get("content") or ""
    return label, answer

# A few shoppers get routed; note the assignment is sticky per shopper id.
for sid in ["shopper-1001", "shopper-1002", "shopper-1003"]:
    label, answer = ask_ab(sid, "Is ground beef in stock, and what aisle?")
    print(f"{sid} -> variant {label}\n    {answer[:160]}\n")

### 6.4 A tiny routing UI

A minimal in-notebook console that mimics the shopper-facing app: type a message, pick how it's routed (auto-split, or force A/B to compare side by side), and it calls the assigned assistant. In a real product this logic sits in your web/app router — the notebook version just makes the routing visible.

*(Falls back to a plain function call if `ipywidgets` isn't available.)*

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display, Markdown

    msg = W.Text(value="Is ground beef in stock, and what aisle?",
                 description="Shopper:", layout=W.Layout(width="70%"))
    shopper = W.Text(value="shopper-1001", description="ID:", layout=W.Layout(width="40%"))
    mode = W.ToggleButtons(options=["auto (hash split)", "force A", "force B", "compare A vs B"],
                           description="Route:")
    go = W.Button(description="Send", button_style="primary")
    out = W.Output()

    def _run(label, text):
        result = client.runs.wait(
            None, variants[label],
            input={"messages": [{"role": "user", "content": text}]},
            metadata={"variant": label, "shopper_id": shopper.value, "experiment": "shopping-prompt-ab"},
        )
        return result["messages"][-1].get("content") or ""

    def _on_click(_):
        out.clear_output()
        with out:
            text = msg.value
            if mode.value == "compare A vs B":
                for label in ("A", "B"):
                    display(Markdown(f"**Variant {label}**\n\n{_run(label, text)}"))
            else:
                label = ("A" if mode.value == "force A"
                         else "B" if mode.value == "force B"
                         else pick_variant(shopper.value))
                display(Markdown(f"**Routed to variant {label}** (assistant `{variants[label]}`)\n\n{_run(label, text)}"))

    go.on_click(_on_click)
    display(W.VBox([msg, shopper, mode, go, out]))
except Exception as e:
    print(f"(ipywidgets UI unavailable: {e})\nFalling back to a direct call:")
    label, answer = ask_ab("shopper-1001", "Is ground beef in stock, and what aisle?")
    print(f"routed to {label}: {answer[:200]}")

### 6.5 Measure the experiment

Every run is tagged two ways — its `assistant_id` (the variant) and `metadata.variant` — so you can slice results by either. In LangSmith, open the project and **group by** `assistant_id` or `metadata.variant` to compare A vs B on latency, cost, and any eval score (correctness, groundedness) you attached in the LangSmith module. From the SDK you can pull the runs back by their metadata:

In [ ]:
from collections import Counter

# Count how the recent A/B runs split across variants (grouping by metadata.variant).
# `client` here is the LangSmith tracing client if you have one imported; the
# LangGraph SDK also lets you filter runs on the deployment by metadata.
counts = Counter()
for sid in [f"shopper-{i}" for i in range(1000, 1020)]:
    counts[pick_variant(sid)] += 1
print("Assignment split across 20 shoppers (stable hash):", dict(counts))

print("\nIn LangSmith, group the project by `metadata.variant` (or `assistant_id`)")
print("to compare A vs B on latency, tokens, and eval scores side by side.")

> **Cleanup (optional).** The two assistants persist on the server so you can keep experimenting. Remove them with `client.assistants.delete(variants["A"])` / `["B"]` when you're done.

## 7. Events out — get notified when things happen

So far the notebook has *called into* the server. Just as important is the server (and LangSmith) **calling out** to your systems when something happens — that's what lets an agent close a loop with the rest of your platform (kick off a fulfillment job, page an on-call, trigger a governance pipeline).

There are **four distinct mechanisms**, and it's worth keeping them straight because they fire on different things and live in different places:

| # | Mechanism | Fires on | Where it's configured |
|---|---|---|---|
| 7.1 | **Run-level webhook** | a single run finishing | passed to `runs.create(...)` (LangGraph SDK) |
| 7.2 | **Automation rules on trace data** | runs matching a filter (batched) | a rule on a LangSmith tracing project |
| 7.3 | **Alerts on thresholds/changes** | error rate / latency / feedback crossing a threshold | LangSmith project alerts |
| 7.4 | **Engine & Context Hub webhooks** | issues, failed agent runs, prompt/skill commits | LangSmith Engine / Context Hub settings |

7.1 runs against the **local dev server** below. 7.2–7.4 are **LangSmith cloud** features that act on your *deployment's tracing project* / Engine / Context Hub — they don't have anything to act on against `langgraph dev`, so we show the exact config + payload shapes and link the docs.

### 7.1 Run-level webhook — "trigger things when a run is done"

Pass a `webhook` URL when you create a run; when that run finishes, the server **POSTs the run object** to your URL. This is the "do something when this specific job is done" hook — e.g. a shopper's big meal-plan + route build completes, and you POST to your app so it notifies them or kicks off order fulfillment.

> **Local gotcha:** the dev server blocks webhook URLs on the **loopback range** (`127.0.0.1` / `localhost`) as an SSRF guard, so a webhook to your own laptop is rejected. Point it at a public capture URL instead — grab a free one from [webhook.site](https://webhook.site) and set `WEBHOOK_URL` (env var or edit the cell). A cloud LangSmith Deployment can reach your real internal endpoints.

In [ ]:
# Set WEBHOOK_URL to a public endpoint (e.g. from https://webhook.site).
# Loopback (127.0.0.1/localhost) is blocked by the dev server's SSRF guard.
webhook_url = os.environ.get("WEBHOOK_URL", "").strip()

if not webhook_url:
    print("No WEBHOOK_URL set — skipping the live fire.")
    print("Grab a URL from https://webhook.site, then set WEBHOOK_URL and re-run.")
    print("\nThe call would be:")
    print('  client.runs.create(thread_id, assistant_id, input={...}, webhook="https://...")')
else:
    # Use a prompt that completes on its own (no file write) so the run reaches a
    # terminal state and the webhook fires. A prompt that trips the write_file
    # interrupt would PAUSE instead of finishing, so no webhook would be sent.
    wh_thread = client.threads.create()
    wh_run = client.runs.create(
        wh_thread["thread_id"],
        assistant_id,
        input={"messages": [{"role": "user", "content": (
            "What aisle is the salsa in, and how much is it?"
        )}]},
        webhook=webhook_url,     # <-- server POSTs the finished run object here
    )
    print("Run created:", wh_run["run_id"], "-> status", wh_run["status"])
    print("Waiting for it to finish (the webhook fires on completion)...")

    # The webhook is sent when the run reaches a terminal state, a few seconds
    # after create() returns. Wait for that so you know when to check your URL.
    status = wh_run["status"]
    for _ in range(60):
        status = client.runs.get(wh_thread["thread_id"], wh_run["run_id"])["status"]
        if status not in ("pending", "running"):
            break
        time.sleep(1)

    print("Final run status:", status)
    if status == "success":
        print("Webhook POSTed the run object to:", webhook_url)
        print("Refresh your webhook.site page — you should see the POST now.")
    elif status == "interrupted":
        print("Run PAUSED on an interrupt (e.g. a file write awaiting approval),")
        print("so it never reached a terminal state and no webhook was sent.")
    else:
        print("Run did not succeed (status above) — no webhook sent.")

### 7.2 Automation rules on trace data

A **rule** watches a tracing project: `filter` + `sampling_rate` decide which runs match, then an **action** runs — add to a dataset, add to an annotation queue, run an online evaluator, and/or **fire a webhook**. This is the fan-out for production traffic (vs. 7.1, which is one run you launched yourself).

Two things to know before you demo it:

- **Payloads are batched per polling window** — the webhook receives a batch of matched runs, not one POST per run.
- **A webhook can fire *before* evaluator scoring lands.** If your consumer needs the feedback score, either add a **feedback filter** (`eq(feedback_key, "correctness")`) so the rule only matches once the score exists, or **split into two rules** (one to score, one to notify on the scored feedback). Belt-and-braces.

The `create_run_rule` helper (`utils/langsmith_rules.py`, from the LangSmith module) now takes `webhook_urls=[...]`. This acts on a **LangSmith tracing project**, so it targets your deployment's project — not the local dev server.

In [ ]:
# This targets a LangSmith *tracing project* (your deployment's), so it needs a
# LangSmith client + project name. Shown as the exact call; guarded so it no-ops
# cleanly if you don't have a project/webhook handy in this session.
from langsmith import Client as LangSmithClient
from utils.langsmith_rules import create_run_rule

LS_PROJECT = os.environ.get("LANGSMITH_PROJECT", "modular-workshops-deep-agent")
notify_url = os.environ.get("WEBHOOK_URL", "").strip()

if not (os.environ.get("LANGSMITH_API_KEY") and notify_url):
    print("Skipping live rule creation (needs LANGSMITH_API_KEY + WEBHOOK_URL).")
    print("The call would be:\n")
    print('  ls = LangSmithClient()')
    print('  create_run_rule(')
    print('      ls, project_name=LS_PROJECT,')
    print('      display_name="notify-out-of-stock",')
    print('      # Only notify once the correctness score exists (avoids firing before scoring):')
    print('      filter=\'and(eq(is_root, true), eq(feedback_key, "correctness"))\',')
    print('      sampling_rate=1.0,')
    print('      webhook_urls=[notify_url],')
    print('  )')
else:
    ls = LangSmithClient()
    rule = create_run_rule(
        ls,
        project_name=LS_PROJECT,
        display_name="notify-scored-runs",
        # Feedback filter so the webhook fires AFTER the correctness score lands.
        filter='and(eq(is_root, true), eq(feedback_key, "correctness"))',
        sampling_rate=1.0,
        webhook_urls=[notify_url],
    )
    print("Rule created:", rule["id"])
    print("Open in UI:", rule["url"])

### 7.3 Alerts on thresholds and changes

Rules act on *individual runs*; **alerts** act on *aggregate metrics* over a time window. LangSmith can alert when **error rate**, **latency (P50/P99)**, or a **feedback score** crosses a threshold — or changes sharply — over a window of up to **60 minutes**, and route the alert to **PagerDuty, Dynatrace, Slack, or any HTTP endpoint** (webhook).

For the shopping assistant, useful alerts might be:

- **Error rate > 5%** over 15 min → page on-call (the store lookup / inventory API may be down).
- **P99 latency > 20s** over 15 min → Slack the team (runs are stalling).
- **`correctness` feedback avg < 0.7** over 60 min → notify (quality regression, e.g. a bad prompt promote from §6).

Alerts are configured per **tracing project** in **LangSmith → your project → Alerts** (metric, threshold, window, destination). They're a cloud feature on your deployment's project, so there's nothing to fire against `langgraph dev` — set them up in the UI on the deployed project. See the [Alerts docs](https://docs.langchain.com/langsmith/alerts).

### 7.4 Engine & Context Hub webhooks

The last set fires from **LangSmith Engine** (Module 5) and the **Context Hub** (Module: LangSmith / composition) — higher-level events about *issues* and *prompt/context changes*, not raw runs.

**Engine webhooks** — subscribe to issue lifecycle events, optionally gated by a severity threshold:

| Event | Fires when |
|---|---|
| `issue.created` | Engine files a new issue (e.g. the broken store-lookup issue from Module 5) |
| `issue.trace.added` | another failing trace is linked to an existing issue |
| `issue.agent_run.failed` | an Engine agent run on the issue fails (with a severity threshold) |

Wire these to your tracker so a `High`-severity `issue.created` opens a ticket or pages the on-call automatically.

**Context Hub webhook** — a **commit event fires whenever an agent or skill file changes** in a Context Hub repo (e.g. someone edits `AGENTS.md` or a `SKILL.md`, or promotes a new assistant prompt version from §6). This is the **governance hook** the platform team will care about: a prompt change can trigger *their* pipeline — review, approval, redeploy, changelog — without engineering in the loop.

Both are configured in **LangSmith settings** (Engine webhooks on the project/Engine; Context Hub webhooks on the repo), not from this notebook. Docs: [Engine webhooks](https://docs.langchain.com/langsmith/engine-webhooks) · [Context Hub](https://docs.langchain.com/langsmith/context-hub).

## Recap

Every feature here came from the **server**, not from changes to the agent — the same `deep_agent` graph you deployed in the deploy module.

| Feature | SDK call | Why it matters for a shopping assistant |
|---|---|---|
| **Background jobs** | `client.runs.create(...)` then `runs.join(...)` | Kick off a long meal-plan + route build without blocking the shopper |
| **Crons** | `client.crons.create(assistant_id, schedule=...)` | Nightly restock checks / weekly staples lists, no external scheduler |
| **Durable execution** | runs execute on a `thread`; `threads.get_state(...)` | Survive disconnects; follow-ups remember the earlier plan |
| **Streaming** | `client.runs.stream(..., stream_mode=...)` | Show live progress ("found salsa in aisle 6") instead of a spinner |
| **HITL** | interrupt -> `Command(resume=...)` | Pause on file writes for a human OK, then resume from the checkpoint |
| **A/B testing** | two `assistants.create(...)` on one graph + your router | Ship a prompt variation with no redeploy; a non-engineer owns the prompt; version with `set_latest` |
| **Events out** | `runs.create(..., webhook=...)`; automation rules; alerts; Engine/Context Hub webhooks | Notify when a run finishes, batch-notify on trace filters, page on threshold breaches, trigger a governance pipeline on prompt changes |

**The same code works against a cloud LangSmith Deployment** — swap `BASE_URL` for the deployment URL and everything above runs unchanged.

**Docs:** [LangGraph Server](https://docs.langchain.com/langgraph-platform/langgraph-server) · [SDK](https://docs.langchain.com/langgraph-platform/sdk) · [Assistants](https://docs.langchain.com/langgraph-platform/assistants) · [Crons](https://docs.langchain.com/langgraph-platform/cron-jobs) · [HITL](https://docs.langchain.com/langgraph-platform/add-human-in-the-loop) · [Streaming](https://docs.langchain.com/langgraph-platform/streaming) · [Webhooks](https://docs.langchain.com/langgraph-platform/use-webhooks) · [Rules](https://docs.langchain.com/langsmith/rules) · [Alerts](https://docs.langchain.com/langsmith/alerts) · [Engine webhooks](https://docs.langchain.com/langsmith/engine-webhooks)

### Teardown

If this notebook started the server, stop it. (If you were running your own `langgraph dev`, this leaves it alone.)

In [ ]:
if dev_proc is not None and dev_proc.poll() is None:
    dev_proc.terminate()
    try:
        dev_proc.wait(timeout=10)
    except Exception:
        dev_proc.kill()
    print("Stopped the langgraph dev server this notebook started.")
else:
    print("No server to stop (either reused an existing one, or it already exited).")